# Gradient boosting for regression

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
A random forest builds its trees **in parallel and independently**. Boosting builds them **one after another**: each new tree learns to fix the **remaining error** of the previous ones. Like a student who, after each exam, practices only their mistakes.

## The algorithm
- Start: $F_0(x)=\bar y$ (the mean).
- For $m=1,\dots,M$:
- Compute the residuals: $r_i=y_i-F_{m-1}(x_i)$.
- Train a small tree $h_m$ on $(x_i, r_i)$.
- Update: $F_m(x)=F_{m-1}(x)+\eta\,h_m(x)$.

$\eta$ is the **learning rate** (say 0.1): each tree makes only a small part of the correction. Why "gradient"? Because the residual is exactly the negative gradient of the squared error, so boosting is gradient descent in the space of functions.

## A worked example
$y=[3, 5, 10]$, $\eta=0.5$:

| Stage | Prediction | Residual |
|---|---|---|
| $F_0$ | [6, 6, 6] | [−3, −1, 4] |
| Tree 1 learns | [−2, −2, 4] | |
| $F_1=F_0+0.5h_1$ | [5, 5, 8] | [−2, 0, 2] |
| Tree 2 ... | | smaller and smaller |

> [After 1 and 20 trees, and the train and test error against the number of trees. The test error rises after a point.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/gbr_stages.png)

*After 1 and 20 trees, and the train and test error against the number of trees. The test error rises after a point.*

[[lab:reg_gbr]]

## Hyperparameters and their balance
| Hyperparameter | Typical | Note |
|---|---|---|
| `n_estimators` | 100 to 1000 | Unlike a forest, too many trees overfit |
| `learning_rate` | 0.01 to 0.1 | Smaller = more trees needed, usually more accurate |
| `max_depth` | 2 to 5 | **Small** trees |
| `subsample` | 0.5 to 1 | Below 1 = stochastic boosting, more robust |
| `n_iter_no_change` | e.g. 10 | **Early stopping** on validation data |

> The golden rule: lower the learning rate and find n_estimators with early stopping.



In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

# Gradient boosting written by hand, on a noisy sine
rng = np.random.default_rng(0)
x = rng.uniform(0, 1, 80)[:, None]; y = np.sin(2 * np.pi * x.ravel()) + rng.normal(0, 0.2, 80)
xt = rng.uniform(0, 1, 300)[:, None]; yt = np.sin(2 * np.pi * xt.ravel()) + rng.normal(0, 0.2, 300)

lr, F, Ft = 0.1, np.full(80, y.mean()), np.full(300, y.mean())
for m in range(1, 301):
    tree = DecisionTreeRegressor(max_depth=2).fit(x, y - F)   # learn the residuals
    F += lr * tree.predict(x); Ft += lr * tree.predict(xt)
    if m in (1, 10, 50, 100, 300):
        print(f"trees={m:3d}  train MSE={np.mean((y - F) ** 2):.3f}  test MSE={np.mean((yt - Ft) ** 2):.3f}")



## Faster versions
`HistGradientBoostingRegressor` bins the features into histograms and is very fast on big data; it also accepts missing values directly. XGBoost, LightGBM and CatBoost follow the same idea and win many competitions on tabular data (run those in Colab).



In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score

X, y = load_diabetes(return_X_y=True)
gb = GradientBoostingRegressor(n_estimators=2000, learning_rate=0.02, max_depth=2, subsample=0.8,
                               validation_fraction=0.2, n_iter_no_change=20, random_state=0).fit(X, y)
print("Trees chosen by early stopping:", gb.n_estimators_)
for name, m in [("GradientBoosting", GradientBoostingRegressor(n_estimators=gb.n_estimators_, learning_rate=0.02, max_depth=2, subsample=0.8, random_state=0)),
                ("HistGradientBoosting", HistGradientBoostingRegressor(learning_rate=0.05, max_depth=3, random_state=0))]:
    print(name, round(cross_val_score(m, X, y, cv=5, scoring="r2").mean(), 3))



| Random forest | Gradient boosting |
|---|---|
| Deep, independent trees | Small trees, one after another |
| Reduces variance | Reduces bias |
| Little tuning | Needs more careful tuning |
| Hard to overfit | Overfits with too many trees |

### Exercise

In the first code, try learning_rate 1.0 and 0.01. Which overfits sooner?

In [ ]:
# Your answer here